In [ ]:
import os

import numpy as np              
import pandas as pd              

import scipy.stats as stats   

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.neighbors import NearestNeighbors, KDTree               
from sklearn.neural_network import MLPClassifier

from sklearn.ensemble import BaggingClassifier, AdaBoostClassifier   

from sklearn.metrics import roc_auc_score, recall_score, f1_score, confusion_matrix   
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold

from imblearn.over_sampling import SMOTE, ADASYN, BorderlineSMOTE 
from imblearn.combine import SMOTETomek, SMOTEENN 

import matplotlib.pyplot as plt
import seaborn as sns

from resampling.load.store import load_datasets_csv

***
Function for outputing complexities of the datasets
***

In [ ]:
def calculate_complexity(X, y, k=5):
    """
    Calculates Maj_Com and Min_Com based on Eq. 15.
    Metric: Average proportion of k-nearest neighbors belonging to the same class.
    """
    if not isinstance(X, pd.DataFrame):
        X = pd.DataFrame(X)
        
    X_encoded = pd.get_dummies(X, drop_first=True)
    X_encoded = X_encoded.astype(float)
    X_norm = (X_encoded - X_encoded.mean()) / X_encoded.std()
    X_norm = X_norm.fillna(0) 
    
    knn = NearestNeighbors(n_neighbors=k+1).fit(X_norm) # +1 because sample includes itself

    def get_class_complexity(target_class):
        class_indices = np.where(y == target_class)[0]
        if len(class_indices) == 0: return 0.0
        distances, indices = knn.kneighbors(X_norm.iloc[class_indices])
        
        neighbor_indices = indices[:, 1:]
        neighbor_labels = y.iloc[neighbor_indices.flatten()].values.reshape(neighbor_indices.shape)
        matches = (neighbor_labels == target_class).astype(int)
    
        return np.mean(np.mean(matches, axis=1))

    maj_com = get_class_complexity(-1)
    min_com = get_class_complexity(1)
    
    return maj_com, min_com

***
Function for outputing complexities of the datasets
***

In [ ]:
def compare_complexities(datasets):
    results = []
    names = []
    
    for name, (X, y) in datasets.items():
        maj_c, min_c = calculate_complexity(X, y)
        results.append([maj_c, min_c])
        names.append(name)
    
    df_res = pd.DataFrame(results, columns=['Maj_Com', 'Min_Com'], index=names)
    
    fig, ax = plt.subplots(figsize=(14, 6))
    
    x = np.arange(len(names))
    width = 0.35
    
    rects1 = ax.bar(x - width/2, df_res['Maj_Com'], width, label='Maj_Com', color='blue')
    rects2 = ax.bar(x + width/2, df_res['Min_Com'], width, label='Min_Com', 
                    color='white', edgecolor='red', hatch='///')
    
    ax.set_ylabel('Data Complexity Measure')
    ax.set_title('Comparison of Majority and Minority Class Complexity')
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=45, ha='right')
    ax.set_ylim(0, 1.05)
    ax.legend()
    
    plt.tight_layout()
    plt.show()
    plt.clf()

In [ ]:
def calculate_g_mean(y_true, y_pred):
    """G-mean = sqrt(Sensitivity * Specificity)"""
    # handle potential missing classes in small folds
    try:
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[-1, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        return np.sqrt(sensitivity * specificity)
    except ValueError:
        return 0.0

In [ ]:
def get_classifier(n_features, n_classes=2):
    """MLP settings from Section 5.3.1"""
    n_input = n_features
    n_output = n_classes
    n_hidden = int(np.ceil(2 * (n_input + n_output) / 3))
    return MLPClassifier(hidden_layer_sizes=(n_hidden,), max_iter=500, random_state=42)

In [ ]:
def run_research_experiments(datasets_dict, re_sc_func):    
    methods = {
        'Original': None,
        'SMOTE': SMOTE(k_neighbors=5, random_state=42),
        'SMOTE-TL': SMOTETomek(smote=SMOTE(k_neighbors=5), random_state=42),
        'SMOTE-ENN': SMOTEENN(smote=SMOTE(k_neighbors=5), random_state=42),
        'BL-SMOTE': BorderlineSMOTE(k_neighbors=5, kind='borderline-1', random_state=42),
        'ADASYN': ADASYN(n_neighbors=5, random_state=42),
        'Re-SC': 'custom' # Flag to use the passed function
    }
    
    results = {metric: {m: [] for m in methods} for metric in ['AUC', 'Recall', 'G-mean', 'F1']}
    dataset_names = []

    print(f"Running Experiments on {len(datasets_dict)} datasets...")

    for name, (X, y) in datasets_dict.items():
        print(f"Processing {name}...", end=" ")
        dataset_names.append(name)

        imputer = SimpleImputer(strategy='mean')
        scaler = StandardScaler()
        
        if hasattr(X, 'values'): X = X.values
        if hasattr(y, 'values'): y = y.values
        
        X_proc = scaler.fit_transform(imputer.fit_transform(X))
        y = y.astype(int) 
        
        ds_res = {m: {'AUC':[], 'Recall':[], 'G-mean':[], 'F1':[]} for m in methods}
        
        rskf = RepeatedStratifiedKFold(n_splits=10, n_repeats=10, random_state=42)
        
        for fold_idx, (train_idx, test_idx) in enumerate(rskf.split(X_proc, y)):
            X_train, X_test = X_proc[train_idx], X_proc[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]
            
            for method_name, sampler in methods.items():
                try:
                    if method_name == 'Re-SC':
                        X_res, y_res = re_sc_func(X_train, y_train)
                        curr_X_test = np.hstack([X_test, X_test])
                        
                    elif method_name == 'Original':
                        X_res, y_res = X_train, y_train
                        curr_X_test = X_test
                        
                    else:
                        X_res, y_res = sampler.fit_resample(X_train, y_train)
                        curr_X_test = X_test

                    clf = get_classifier(X_res.shape[1])
                    clf.fit(X_res, y_res)
                    
                    y_pred = clf.predict(curr_X_test)

                    if hasattr(clf, "predict_proba"):
                        if len(clf.classes_) == 2:
                            pos_idx = np.where(clf.classes_ == 1)[0][0]
                            y_proba = clf.predict_proba(curr_X_test)[:, pos_idx]
                        else:
                            y_proba = y_pred
                    else:
                        y_proba = y_pred

                    ds_res[method_name]['AUC'].append(roc_auc_score(y_test, y_proba))
                    ds_res[method_name]['Recall'].append(recall_score(y_test, y_pred, pos_label=1))
                    ds_res[method_name]['F1'].append(f1_score(y_test, y_pred, pos_label=1))
                    ds_res[method_name]['G-mean'].append(calculate_g_mean(y_test, y_pred))
                    
                except Exception:
                    pass

        for m in methods:
            if ds_res[m]['AUC']:
                results['AUC'][m].append(np.mean(ds_res[m]['AUC']))
                results['Recall'][m].append(np.mean(ds_res[m]['Recall']))
                results['F1'][m].append(np.mean(ds_res[m]['F1']))
                results['G-mean'][m].append(np.mean(ds_res[m]['G-mean']))
            else:
                for metric in results: results[metric][m].append(np.nan)
                
        print("Done.")

    return results, dataset_names

In [ ]:
def generate_result_tables(results, dataset_names):
    final_tables = {}
    for metric, data in results.items():
        df = pd.DataFrame(data, index=dataset_names)
        df.loc['Average'] = df.mean()
        df.loc['Average Rank'] = df.iloc[:-1].rank(axis=1, ascending=False).mean()
        final_tables[metric] = df
        print(f"\n--- {metric} Results ---")
        print(df)
    return final_tables

In [ ]:
def generate_ttest_table(tables):
    summary = []
    resc_col = 'Re-SC'
    
    for metric, df in tables.items():
        row_res = {'Metric': metric}
        
        for method in df.columns:
            if method == resc_col: continue

            data_rows = df.iloc[:-2] 
            
            vec_resc = data_rows[resc_col]
            vec_comp = data_rows[method]
            
            wins = (vec_resc > vec_comp).sum()
            losses = (vec_resc < vec_comp).sum()
            ties = (vec_resc == vec_comp).sum()
            
            row_res[method] = f"{wins}-{ties}-{losses}"
            
        summary.append(row_res)
        
    return pd.DataFrame(summary)

In [ ]:
def save_experiment_results(tables_dict, folder_path="./"):
    """
    Saves the generated experimental tables to CSV files.
    
    Args:
        tables_dict (dict): Dictionary containing DataFrames for AUC, Recall, G-mean, F1.
        summary_table (pd.DataFrame): The Win-Tie-Lose summary table.
        folder_path (str): Directory to save the files.
    """
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
        print(f"Created directory: {folder_path}")

    map_names = {
        'AUC': 'Table2_AUC.csv',
        'Recall': 'Table3_Recall.csv',
        'G-mean': 'Table4_Gmean.csv',
        'F1': 'Table5_F1.csv'
    }
    
    for metric_name, df in tables_dict.items():
        filename = map_names.get(metric_name, f"Table_{metric_name}.csv")
        file_path = os.path.join(folder_path, filename)

        df.to_csv(file_path)
        print(f"Saved: {file_path}")

    
    print("\nAll tables saved successfully.")

# ==========================================
# Usage Example
# ==========================================
# Assuming you have run the previous code:
# final_tables = generate_result_tables(results, ds_names)
# table_6 = generate_ttest_table(final_tables)

In [ ]:
datasets=load_datasets_csv('../resampling/data/resampled')

In [ ]:
raw_results, ds_names = run_research_experiments(datasets, my_re_sc_wrapper)

tables = generate_result_tables(raw_results, ds_names)

auc_table = tables['AUC']
recall_table = tables['Recall']

In [ ]:
save_experiment_results(tables)